# Arabic Math Reasoning LLM — Full Pipeline Demo

## Build a Large Language Model (From Scratch) — Sebastian Raschka

This notebook walks through **every step** of building an Arabic LLM from scratch,
following all 7 chapters of the book, applied to **math reasoning** using the
**Arabic-gsm8k-v2** dataset.

---

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import json
from pathlib import Path

PROJECT_ROOT = Path('..').resolve()

DEVICE = torch.device(
    'cuda' if torch.cuda.is_available() else
    'mps' if torch.backends.mps.is_available() else 'cpu'
)
print(f'Using device: {DEVICE}')

---
# Chapter 1: Understanding Large Language Models

**Key concepts**: Next-token prediction, tokens & embeddings, Transformer intuition,
GPT overview, scaling laws.

LLMs are **autoregressive models** that predict the next token given all previous tokens.
They learn the statistical patterns of language through massive-scale training.

```
P(token_t | token_1, token_2, ..., token_{t-1})
```

Our model: A **GPT (Generative Pre-trained Transformer)** trained on Arabic math problems.

---
# Chapter 2: Working With Text Data

**Key concepts**: Tokenization, vocabulary building, encoding/decoding,
PyTorch Dataset & DataLoader, batching.

## Step 1: Download & Prepare the Arabic GSM8K Dataset

In [ ]:
from src.training.prepare_data import download_arabic_gsm8k, train_tokenizer

# Download and process the dataset
info = download_arabic_gsm8k()
print(f"\nDataset ready:")
print(f"  Pretraining text: {info['pretrain_path']}")
print(f"  SFT pairs: {info['sft_path']}")
print(f"  Total examples: {info['num_examples']}")

## Step 2: Build BPE Tokenizer from Scratch

**Byte-Pair Encoding (BPE)** iteratively merges the most frequent character pairs
to build a vocabulary of sub-word tokens.

```
1. Start with characters: ['ك', 'ت', 'ا', 'ب']
2. Count pairs: ('ك','ت')=100, ('ت','ا')=200, ('ا','ب')=150
3. Merge most frequent: ('ت','ا') → 'تا'
4. Repeat until vocab_size reached
```

In [ ]:
tokenizer = train_tokenizer(info['pretrain_path'], vocab_size=5000)
print(f"\nTokenizer trained:")
print(f"  Vocabulary size: {tokenizer.vocab_size}")
print(f"  BPE merges: {len(tokenizer.merges)}")

## Step 3: Visualize Tokenization

In [ ]:
test_sentences = [
    'كم عدد التفاحات المتبقية إذا كان لديك ١٠ وأعطيت ٣؟',
    'اشترى محمد ٣ أقلام بسعر ٥ ريالات للقلم الواحد',
    'ما هو حاصل جمع ٢٥ و ١٧؟',
]

print('=== Tokenization Visualization ===')
for sent in test_sentences:
    ids = tokenizer.encode(sent)
    details = tokenizer.get_token_details(sent)
    decoded = tokenizer.decode(ids)
    
    print(f'\n  Original ({len(sent)} chars): {sent}')
    print(f'  Tokens ({len(ids)}): {[d["token"] for d in details]}')
    print(f'  IDs: {ids[:20]}...' if len(ids) > 20 else f'  IDs: {ids}')
    print(f'  Decoded: {decoded}')
    print(f'  Compression: {len(sent)/len(ids):.1f} chars/token')

## Step 4: Build PyTorch Dataset & DataLoader

In [ ]:
from src.training.prepare_data import PretrainDataset, SFTDataset
from torch.utils.data import DataLoader, random_split

# Pretraining dataset (sliding window)
CONTEXT_LENGTH = 256
pretrain_dataset = PretrainDataset(
    info['pretrain_path'], tokenizer, context_length=CONTEXT_LENGTH, stride=128
)
print(f'Pretrain dataset: {len(pretrain_dataset)} samples')

# Show a sample
inp, tgt = pretrain_dataset[0]
print(f'\nSample input shape: {inp.shape}')
print(f'Sample target shape: {tgt.shape}')
print(f'Input tokens (first 20): {inp[:20].tolist()}')
print(f'Target tokens (first 20): {tgt[:20].tolist()}')
print(f'\nNotice: target is shifted by 1 position (next-token prediction)')
print(f'  Input:  [..., token_i, token_{"i+1"}, ...]')
print(f'  Target: [..., token_{"i+1"}, token_{"i+2"}, ...]')

# Split into train/val
val_size = max(1, int(0.1 * len(pretrain_dataset)))
train_size = len(pretrain_dataset) - val_size
train_ds, val_ds = random_split(pretrain_dataset, [train_size, val_size])

BATCH_SIZE = 8
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, drop_last=False)

print(f'\nTrain batches: {len(train_loader)}')
print(f'Val batches: {len(val_loader)}')

# Visualize a batch
batch_inp, batch_tgt = next(iter(train_loader))
print(f'\nBatch input shape: {batch_inp.shape}  (batch_size × context_length)')
print(f'Batch target shape: {batch_tgt.shape}')

---
# Chapter 3: Coding Attention Mechanisms

**Key concepts**: Scaled dot-product attention, QKV math, causal masking,
multi-head attention, Transformer block.

## Step 5: Understand Scaled Dot-Product Attention

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

- **Q** (Query): What am I looking for?
- **K** (Key): What do I contain?
- **V** (Value): What information do I provide?

In [ ]:
from src.model.attention import scaled_dot_product_attention, MultiHeadAttention

# Demonstrate with small tensors
d_k = 4
seq_len = 5

torch.manual_seed(42)
Q = torch.randn(1, 1, seq_len, d_k)  # (B, heads, T, d_k)
K = torch.randn(1, 1, seq_len, d_k)
V = torch.randn(1, 1, seq_len, d_k)

print('=== QKV Shapes ===')
print(f'Q: {Q.shape}  — queries (what each token looks for)')
print(f'K: {K.shape}  — keys (what each token offers)')
print(f'V: {V.shape}  — values (actual information)')

# Without causal mask
context, weights = scaled_dot_product_attention(Q, K, V)
print(f'\n=== Without Causal Mask ===')
print(f'Attention weights:\n{weights[0, 0].numpy().round(3)}')
print('Each row sums to 1:', weights[0, 0].sum(dim=-1).numpy().round(3))

# With causal mask
causal_mask = torch.tril(torch.ones(seq_len, seq_len)).unsqueeze(0).unsqueeze(0)
context_masked, weights_masked = scaled_dot_product_attention(Q, K, V, mask=causal_mask)
print(f'\n=== With Causal Mask ===')
print(f'Attention weights (causal):\n{weights_masked[0, 0].numpy().round(3)}')
print('Future positions are zeroed out (each token can only attend to past)')

In [ ]:
# Visualize attention patterns
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

labels = [f'pos_{i}' for i in range(seq_len)]

im1 = axes[0].imshow(weights[0, 0].numpy(), cmap='Blues', aspect='auto')
axes[0].set_title('Full Attention (bidirectional)')
axes[0].set_xlabel('Key position')
axes[0].set_ylabel('Query position')
plt.colorbar(im1, ax=axes[0], shrink=0.8)

im2 = axes[1].imshow(weights_masked[0, 0].numpy(), cmap='Blues', aspect='auto')
axes[1].set_title('Causal Attention (autoregressive)')
axes[1].set_xlabel('Key position')
axes[1].set_ylabel('Query position')
plt.colorbar(im2, ax=axes[1], shrink=0.8)

plt.suptitle('Scaled Dot-Product Attention — Chapter 3', fontsize=14)
plt.tight_layout()
plt.savefig('../results/plots/attention_patterns.png', dpi=150, bbox_inches='tight')
plt.show()

print('Left: Every token attends to all positions')
print('Right: Each token can only attend to itself and earlier positions')
print('       (upper triangle is masked to -inf before softmax)')

## Step 6: Multi-Head Attention

Instead of one attention operation, we run **multiple heads in parallel**,
each with its own Q, K, V projections, then concatenate and project.

$$\text{MultiHead}(Q,K,V) = \text{Concat}(\text{head}_1, ..., \text{head}_h) W^O$$

In [ ]:
d_model = 64
n_heads = 4
context_length = 16

mha = MultiHeadAttention(d_model, n_heads, context_length)
print(f'=== Multi-Head Attention ===')
print(f'd_model: {d_model}')
print(f'n_heads: {n_heads}')
print(f'd_k per head: {d_model // n_heads}')
print(f'Parameters: {sum(p.numel() for p in mha.parameters()):,}')

# Forward pass
x = torch.randn(2, 10, d_model)  # (batch=2, seq_len=10, d_model=64)
output, attn_weights = mha(x, return_attention=True)

print(f'\nInput shape:  {x.shape}')
print(f'Output shape: {output.shape}  (same as input — residual-ready)')
print(f'Attention weights shape: {attn_weights.shape}  (B, n_heads, T, T)')

# Visualize all heads
fig, axes = plt.subplots(1, n_heads, figsize=(4*n_heads, 4))
for h in range(n_heads):
    im = axes[h].imshow(attn_weights[0, h].detach().numpy(), cmap='Blues', aspect='auto')
    axes[h].set_title(f'Head {h}')
    axes[h].set_xlabel('Key')
    if h == 0:
        axes[h].set_ylabel('Query')
plt.suptitle('Multi-Head Attention Patterns (different heads learn different patterns)', fontsize=12)
plt.tight_layout()
plt.savefig('../results/plots/multi_head_attention.png', dpi=150, bbox_inches='tight')
plt.show()

---
# Chapter 4: Implementing a GPT Model From Scratch

**Key concepts**: Token & positional embeddings, stacked Transformer blocks,
residual connections, layer norm, output head, sampling strategies.

## Step 7: Build the GPT Model

In [ ]:
from src.model.transformer import GPTModel, GPT_CONFIG, LayerNorm, FeedForward, TransformerBlock

# Configure model
cfg = dict(GPT_CONFIG)
cfg['vocab_size'] = tokenizer.vocab_size
cfg['context_length'] = CONTEXT_LENGTH

print('=== GPT Configuration ===')
for k, v in cfg.items():
    print(f'  {k}: {v}')

# Build model
model = GPTModel(cfg)
print(f'\nTotal parameters: {model.count_parameters():,}')

# Layer breakdown
print(f'\n=== Layer-by-Layer Breakdown ===')
total_params = 0
for info in model.get_layer_info():
    print(f"  {info['name']:30s} | {info['shape']:30s} | {info['params']:>10,} params")
    total_params += info['params']

## Step 8: Visualize Embeddings

In [ ]:
# Token embeddings (before training — random)
sample_text = 'كم عدد التفاحات'
sample_ids = tokenizer.encode(sample_text)
sample_tensor = torch.tensor([sample_ids])

with torch.no_grad():
    tok_emb = model.tok_emb(sample_tensor)  # (1, T, emb_dim)
    pos_emb = model.pos_emb(torch.arange(len(sample_ids)))  # (T, emb_dim)
    combined = tok_emb[0] + pos_emb  # (T, emb_dim)

print(f'Input: "{sample_text}"')
print(f'Token IDs: {sample_ids}')
print(f'Token embedding shape: {tok_emb.shape}')
print(f'Positional embedding shape: {pos_emb.shape}')
print(f'Combined shape: {combined.shape}')

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

details = tokenizer.get_token_details(sample_text)
labels = [d['token'] for d in details]

axes[0].imshow(tok_emb[0].numpy()[:, :50], cmap='RdBu', aspect='auto')
axes[0].set_yticks(range(len(labels)))
axes[0].set_yticklabels(labels)
axes[0].set_title('Token Embeddings (first 50 dims)')
axes[0].set_xlabel('Embedding dimension')

axes[1].imshow(pos_emb.numpy()[:, :50], cmap='RdBu', aspect='auto')
axes[1].set_yticks(range(len(labels)))
axes[1].set_yticklabels([f'pos {i}' for i in range(len(labels))])
axes[1].set_title('Positional Embeddings (first 50 dims)')
axes[1].set_xlabel('Embedding dimension')

axes[2].imshow(combined.numpy()[:, :50], cmap='RdBu', aspect='auto')
axes[2].set_yticks(range(len(labels)))
axes[2].set_yticklabels(labels)
axes[2].set_title('Token + Positional (combined input)')
axes[2].set_xlabel('Embedding dimension')

plt.suptitle('Embedding Visualization — Chapter 4', fontsize=14)
plt.tight_layout()
plt.savefig('../results/plots/embeddings.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 9: Forward Pass Visualization

In [ ]:
# Forward pass through the model (before training)
with torch.no_grad():
    logits, attn_maps = model(sample_tensor, return_attention=True)

print(f'Input shape: {sample_tensor.shape}  (batch_size, seq_len)')
print(f'Output logits shape: {logits.shape}  (batch_size, seq_len, vocab_size)')
print(f'Attention maps: {len(attn_maps)} layers, each {attn_maps[0].shape}')

# Show predicted next-token probabilities (before training = random)
probs = torch.softmax(logits[0, -1, :], dim=-1)
top5_probs, top5_ids = torch.topk(probs, 5)

print(f'\nNext-token predictions (untrained model = ~random):')
for prob, tid in zip(top5_probs, top5_ids):
    tok = tokenizer.inverse_vocab.get(tid.item(), '?')
    print(f'  "{tok}" (id={tid.item()}): {prob.item():.4f}')

---
# Chapter 5: Pretraining on Unlabeled Data

**Key concepts**: Self-supervised learning, cross-entropy loss, training loops,
optimization, checkpointing.

The model learns to predict the next token given all previous tokens.
No labels needed — the text itself provides the supervision signal.

## Step 10: Pretrain the Model

In [ ]:
from src.training.pretrain import pretrain, calc_loss_loader

# Move model to device
model = model.to(DEVICE)

# Check initial loss (should be ~ln(vocab_size))
initial_loss = calc_loss_loader(model, val_loader, DEVICE, max_batches=5)
import math
expected_random_loss = math.log(tokenizer.vocab_size)
print(f'Initial loss: {initial_loss:.4f}')
print(f'Expected random loss (ln({tokenizer.vocab_size})): {expected_random_loss:.4f}')
print(f'Initial perplexity: {math.exp(initial_loss):.0f}')

In [ ]:
# Run pretraining
NUM_EPOCHS = 10  # Adjust based on your hardware

pretrain_history = pretrain(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    tokenizer=tokenizer,
    device=DEVICE,
    num_epochs=NUM_EPOCHS,
    lr=5e-4,
    log_interval=50,
    eval_interval=200,
    sample_interval=500,
    sample_prompt='حل المساله الرياضيه',
)

## Step 11: Visualize Pretraining Results

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss curves
if pretrain_history.get('train_losses'):
    axes[0].plot(pretrain_history['train_losses'], 'b-', alpha=0.7, label='Train Loss')
    axes[0].set_xlabel('Log Step')
    axes[0].set_ylabel('Cross-Entropy Loss')
    axes[0].set_title('Training Loss — Chapter 5')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

# LR schedule
if pretrain_history.get('lrs'):
    axes[1].plot(pretrain_history['lrs'], 'g-', alpha=0.7)
    axes[1].set_xlabel('Log Step')
    axes[1].set_ylabel('Learning Rate')
    axes[1].set_title('Cosine LR with Warmup')
    axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../results/plots/pretrain_curves.png', dpi=150, bbox_inches='tight')
plt.show()

# Show samples over training
if pretrain_history.get('samples'):
    print('\n=== Generated Samples During Training ===')
    for s in pretrain_history['samples']:
        print(f"\nStep {s['step']}:")
        print(f"  {s['text'][:200]}")

## Step 12: Generate Text with Pretrained Model

In [ ]:
from src.model.transformer import generate, generate_step_by_step

# Generate from pretrained model
test_prompts = [
    'اذا كان لدى احمد',
    'كم عدد',
    'حل المساله',
]

print('=== Pretrained Model Generations ===')
for prompt in test_prompts:
    ids = tokenizer.encode(prompt)
    inp = torch.tensor([ids], device=DEVICE)
    gen_ids = generate(model, inp, max_new_tokens=60,
                       context_length=CONTEXT_LENGTH, temperature=0.8, top_k=25)
    gen_text = tokenizer.decode(gen_ids[0].tolist())
    print(f'\nPrompt: {prompt}')
    print(f'Generated: {gen_text}')

# Step-by-step generation visualization
print('\n=== Step-by-Step Generation ===')
prompt = 'اذا كان'
ids = tokenizer.encode(prompt)
inp = torch.tensor([ids], device=DEVICE)
steps = generate_step_by_step(model, inp, max_new_tokens=8,
                               context_length=CONTEXT_LENGTH, temperature=0.8)

print(f'Prompt: "{prompt}"\n')
for s in steps:
    tok = tokenizer.inverse_vocab.get(s['token_id'], '?')
    top5 = [(tokenizer.inverse_vocab.get(tid, '?'), f"{p:.3f}")
            for tid, p in zip(s['top5_ids'], s['top5_probs'])]
    print(f"  Step {s['step']}: chose '{tok}' (p={s['probability']:.3f})")
    print(f"    Top-5 candidates: {top5}")

---
# Chapter 6: Fine-Tuning for Classification

**Key concepts**: Transfer learning, classification head, freezing vs full fine-tuning,
evaluation metrics.

We demonstrate adding a classification head on top of the pretrained model.
This shows how the model's learned representations can be repurposed.

In [ ]:
from src.training.finetune import GPTClassifier

# Create a classifier (e.g., difficulty level: easy/medium/hard)
num_classes = 3
classifier = GPTClassifier(model, num_classes=num_classes, freeze_base=True)

total_params = sum(p.numel() for p in classifier.parameters())
trainable_params = classifier.count_trainable_parameters()

print('=== Classification Head (Chapter 6) ===')
print(f'Total parameters: {total_params:,}')
print(f'Trainable parameters: {trainable_params:,}')
print(f'Frozen parameters: {total_params - trainable_params:,}')
print(f'Trainable ratio: {trainable_params/total_params:.1%}')
print(f'\nOnly the last transformer block + classification head are trainable')
print(f'This is "transfer learning" — reuse pretrained knowledge')

---
# Chapter 7: Fine-Tuning to Follow Instructions (SFT)

**Key concepts**: Instruction datasets, prompt formatting, supervised fine-tuning (SFT),
loss masking.

We fine-tune the model to follow math problem instructions using the Arabic GSM8K dataset
with **loss masking**: the loss is only computed on response tokens, not the instruction.

## Step 13: Prepare SFT Data with Loss Masking

In [ ]:
# Build SFT dataset
sft_dataset = SFTDataset(info['sft_path'], tokenizer, context_length=CONTEXT_LENGTH)
print(f'SFT dataset: {len(sft_dataset)} samples')

# Show a sample with loss mask
inp, tgt, mask = sft_dataset[0]
print(f'\nSample shapes:')
print(f'  Input:     {inp.shape}')
print(f'  Target:    {tgt.shape}')
print(f'  Loss mask: {mask.shape}')

# Visualize loss masking
print(f'\n=== Loss Masking Visualization ===')
print(f'Mask values (0=ignore, 1=compute loss):')
prompt_len = (mask == 0).sum().item()
response_len = (mask == 1).sum().item()
print(f'  Prompt tokens (masked, no loss): {prompt_len}')
print(f'  Response tokens (loss computed):  {response_len}')

fig, ax = plt.subplots(figsize=(14, 2))
mask_display = mask.numpy().reshape(1, -1)
ax.imshow(mask_display, cmap='RdYlGn', aspect='auto', vmin=0, vmax=1)
ax.set_xlabel('Token position')
ax.set_yticks([])
ax.set_title('Loss Mask: Red=Instruction (no loss) | Green=Response (loss computed)')
plt.tight_layout()
plt.savefig('../results/plots/loss_masking.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 14: Run Supervised Fine-Tuning

In [ ]:
from src.training.finetune import sft_finetune

# Split SFT data
sft_val_size = max(1, int(0.1 * len(sft_dataset)))
sft_train_size = len(sft_dataset) - sft_val_size
sft_train_ds, sft_val_ds = random_split(sft_dataset, [sft_train_size, sft_val_size])

sft_train_loader = DataLoader(sft_train_ds, batch_size=4, shuffle=True, drop_last=True)
sft_val_loader = DataLoader(sft_val_ds, batch_size=4, shuffle=False, drop_last=False)

print(f'SFT train: {sft_train_size} | SFT val: {sft_val_size}')

# Fine-tune
SFT_EPOCHS = 5
sft_history = sft_finetune(
    model=model,
    train_loader=sft_train_loader,
    val_loader=sft_val_loader,
    tokenizer=tokenizer,
    device=DEVICE,
    num_epochs=SFT_EPOCHS,
    lr=2e-5,
)

## Step 15: Compare Pretrained vs Fine-tuned

In [ ]:
# Visualize SFT loss
fig, ax = plt.subplots(figsize=(10, 5))
if sft_history.get('train_losses'):
    ax.plot(sft_history['train_losses'], 'b-', alpha=0.7, label='SFT Train Loss')
if sft_history.get('val_losses'):
    ax.plot(np.linspace(0, len(sft_history['train_losses']),
            len(sft_history['val_losses'])),
            sft_history['val_losses'], 'r-o', label='SFT Val Loss')
ax.set_xlabel('Step')
ax.set_ylabel('Loss')
ax.set_title('Supervised Fine-Tuning Loss — Chapter 7')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../results/plots/sft_curves.png', dpi=150, bbox_inches='tight')
plt.show()

# Compare samples across epochs
if sft_history.get('samples'):
    print('\n=== SFT Samples Across Epochs ===')
    for s in sft_history['samples']:
        print(f"\nEpoch {s['epoch']}:")
        print(f"  {s['text'][:200]}")

In [ ]:
# Final comparison: generate with fine-tuned model
math_questions = [
    'إذا كان لدى سارة ١٢ كتاباً وأعطت ٤ لصديقتها، كم كتاباً تبقى لديها؟',
    'اشترى أحمد ٣ أقلام بسعر ٥ ريالات للقلم الواحد. كم دفع إجمالاً؟',
    'إذا كان عمر خالد ١٥ سنة وعمر أخيه ضعف عمره، فكم عمر أخيه؟',
]

print('=== Fine-tuned Model: Math Reasoning ===')
for q in math_questions:
    prompt = f'حل المسألة الرياضية التالية خطوة بخطوة.\n{q}'
    ids = tokenizer.encode(prompt)
    inp = torch.tensor([ids], device=DEVICE)
    gen_ids = generate(model, inp, max_new_tokens=100,
                       context_length=CONTEXT_LENGTH, temperature=0.7, top_k=25)
    gen_text = tokenizer.decode(gen_ids[0].tolist())
    print(f'\nQ: {q}')
    print(f'A: {gen_text}')
    print('-' * 60)

---
# Evaluation & Error Analysis

Measuring model quality with perplexity, answer accuracy, and error categorization.

In [ ]:
from src.evaluation.metrics import (
    compute_perplexity, compute_answer_accuracy,
    analyze_errors, llm_judge_score, generation_quality_report
)

# Perplexity
ppl = compute_perplexity(model, val_loader, DEVICE)
print(f'Perplexity: {ppl:.2f}')
print(f'(Lower = better. Random baseline ≈ {tokenizer.vocab_size})')

# Answer accuracy on a sample
with open(info['sft_path'], 'r', encoding='utf-8') as f:
    sft_all = json.load(f)

eval_samples = sft_all[:30]
questions = [s['input'] for s in eval_samples]
answers = [s['output'] for s in eval_samples]

acc = compute_answer_accuracy(model, tokenizer, questions, answers, DEVICE)
print(f'\nAnswer Accuracy: {acc["accuracy"]:.2%} ({acc["correct"]}/{acc["total"]})')

# Error analysis
errors = analyze_errors(acc['results'])
print(f'\nError Categories:')
for cat, cnt in errors['category_counts'].items():
    print(f'  {cat}: {cnt}')

In [ ]:
# Error analysis visualization
cats = errors['category_counts']
if cats:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Pie chart
    labels = list(cats.keys())
    sizes = list(cats.values())
    colors = plt.cm.Set3(np.linspace(0, 1, len(labels)))
    axes[0].pie(sizes, labels=labels, colors=colors, autopct='%1.1f%%', startangle=90)
    axes[0].set_title('Error Category Distribution')
    
    # Bar chart
    axes[1].barh(labels, sizes, color='steelblue')
    axes[1].set_xlabel('Count')
    axes[1].set_title('Error Categories')
    
    plt.tight_layout()
    plt.savefig('../results/plots/error_analysis.png', dpi=150, bbox_inches='tight')
    plt.show()

# Show examples of each error type
print('\n=== Error Examples ===')
for etype, examples in errors.get('examples', {}).items():
    if examples:
        print(f'\n--- {etype.upper()} ---')
        ex = examples[0]
        print(f'  Q: {ex["question"][:100]}')
        print(f'  Gold: {ex["gold_answer"][:100]}')
        print(f'  Pred: {ex["generated"][:100]}')

---
# Attention Visualization on Trained Model

In [ ]:
# Visualize attention on a math problem
viz_text = 'اذا كان لدى احمد خمسه تفاحات واعطى اثنتين'
viz_ids = tokenizer.encode(viz_text)
viz_inp = torch.tensor([viz_ids], device=DEVICE)

with torch.no_grad():
    _, attn_maps = model(viz_inp, return_attention=True)

details = tokenizer.get_token_details(viz_text)
labels = [d['token'][:6] for d in details]

n_layers = len(attn_maps)
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
for idx, ax in enumerate(axes.flat):
    if idx >= n_layers:
        ax.axis('off')
        continue
    attn = attn_maps[idx][0, 0, :len(labels), :len(labels)].cpu().numpy()
    ax.imshow(attn, cmap='Blues', aspect='auto')
    ax.set_xticks(range(len(labels)))
    ax.set_yticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=7)
    ax.set_yticklabels(labels, fontsize=7)
    ax.set_title(f'Layer {idx}, Head 0')

plt.suptitle('Attention Across Layers (Trained Model) — Math Problem', fontsize=14)
plt.tight_layout()
plt.savefig('../results/plots/trained_attention.png', dpi=150, bbox_inches='tight')
plt.show()

---
# Conclusion

## Key Learnings
1. **BPE Tokenization** handles Arabic text effectively by learning sub-word units
2. **Self-Attention** allows tokens to contextualize based on all previous positions
3. **Pretraining** teaches language structure through next-token prediction
4. **SFT with loss masking** teaches the model to follow instructions
5. **Small models** can learn basic patterns but struggle with complex reasoning

## Limitations
- Small model size (~30M params) limits reasoning depth
- Limited training data (Arabic GSM8K ~8K examples)
- BPE vocabulary may not capture all Arabic morphological variations
- No RLHF or preference optimization applied

## Future Work
- Scale model to 100M+ parameters
- Incorporate more Arabic text data for pretraining
- Add RLHF / DPO alignment
- Chain-of-thought prompting for better reasoning
- Multi-task fine-tuning (translation, summarization, QA)